In [ ]:
import sys
import os
from datetime import datetime
import torch

import matplotlib.pyplot as plt

sys.path.append('..')

from PySESM.models.SESM.SESM import SESM_Model
from PySESM.test_functions.MultivariateNormal import MultivariateNormal
from PySESM.base_functions.Function import GaussianFunctions

In [ ]:
def plot_mean_std(data, n_experiments, results_path, parameter):
    means = [torch.mean(data[n, :]).item() for n in range(n_experiments)]
    std_devs = [torch.std(data[n, :]).item() for n in range(n_experiments)]
    
    fig = plt.figure(figsize=(10, 5))
    for i in range(n_experiments):
        plt.scatter(
            means[i],
            std_devs[i],
            color=plt.cm.plasma(i / n_experiments),
            marker='o',
            label=f'Experiment {i+1}'
        )

    plt.title(f'Mean and standard deviation of the {parameter} for each experiment')
    plt.xlabel('Mean')
    plt.ylabel('Standard deviation')

    # Add a legend
    plt.legend()

    plt.savefig(f'{results_path}/{parameter}_mean_std.png')

In [ ]:
def plot_features_times(times, n_features, n_experiments, results_path):
    fig = plt.figure(figsize=(10, 5))
    
    means = [torch.mean(times[n, :]).item() for n in range(n_experiments)]
    
    plt.plot(n_features, means, marker='o')
    
    plt.title(f'Time vs Number of features for each experiment')
    plt.xlabel('Number of features')
    plt.ylabel('Time (s)')
    
    plt.savefig(f'{results_path}/features_times.png')

In [ ]:
# Experiment parameters

n_features =  [   2,    3,    5,   10,   20,    50]
l_functions = [  10,   10,   10,   10,   10,    10]
n_samples =   [1000, 1000, 1000, 1000, 1000,  1000]

n_experiments = len(n_features)
n_repetitions = 10
n_gaussians = 3

search_space = torch.tensor([-3, 3])

In [ ]:
# Model parameters

model_epochs = 50
ista_epochs = 100   
dictionary_epochs = 60

ista_alpha = 0.02
ista_lambd = 0.001

dictionary_alpha = 0.02

In [ ]:
datetime = datetime.now().strftime("%d-%m-%Y@%H:%M")

results_path = f'./results'
if not os.path.exists(results_path):
    os.mkdir(results_path)

results_path = f'./results/{datetime}'
if not os.path.exists(results_path):
    os.mkdir(results_path)

In [ ]:
loss_results = torch.zeros((n_experiments, n_repetitions))
time_results = torch.zeros((n_experiments, n_repetitions))


print(loss_results.shape)
print(time_results.shape)

for n_experiment in range(n_experiments):
    n_samples_ = n_samples[n_experiment]
    n_features_ = n_features[n_experiment]
    l_functions_ = l_functions[n_experiment]
    
    means = [torch.randint(search_space[0], search_space[1], (n_features_,)) for _ in range(n_gaussians)]
    covariances = [torch.eye(n_features_) for _ in range(n_gaussians)]
    scale_factors = [torch.rand(1) for _ in range(n_gaussians)]
    
    plot_n_samples = n_samples_
    plot_samples = torch.tensor([])
    for n in range(n_features_):
        # feature = torch.linspace(search_space[0] - 2, search_space[1] + 2, plot_n_samples)
        feature = torch.rand(plot_n_samples) * (search_space[1] - search_space[0]) + search_space[0]
        # plot_samples.append(feature)
        plot_samples = torch.cat((plot_samples, feature.unsqueeze(1)), dim=1)
    
    mvn = MultivariateNormal(
        n_features=n_features_,
        mus = means,
        covariances=covariances,
        scale_factors=scale_factors
    )
    
    for n_repetition in range(n_repetitions):
        print('-'*50)
        print(f'Experiment {n_experiment + 1}/{n_experiments}, repetition {n_repetition + 1}/{n_repetitions}')
        print(f'Number of samples: {n_samples_}\nNumber of features: {n_features_}\nNumber of functions: {l_functions_}')
        print('-'*50)
            
        X, y = mvn.sample_n(n_samples_, search_space)
        
        gaussian_function = GaussianFunctions(n_features=n_features_, n_functions=l_functions_)
        
        model = SESM_Model(
            n_samples=n_samples_,
            n_features=n_features_,
            n_functions=l_functions_,
            psi=gaussian_function.gaussian
        )
        
        model.fit(
            X=X,
            y=y,
            model_epochs=model_epochs,
            ista_epochs=ista_epochs,
            ista_alpha=ista_alpha,
            ista_lambd=ista_lambd,
            dictionary_epochs=dictionary_epochs,
            dictionary_alpha=dictionary_alpha
        )
        
        experiment_path = f'{results_path}/Experiment_{n_experiment+1}'
        if(not os.path.exists(experiment_path)):
            os.mkdir(experiment_path)
            
        experiment_path = f'{experiment_path}/{n_experiment+1}.{n_repetition+1}'
        if(not os.path.exists(experiment_path)):
            os.mkdir(experiment_path)
        
        original_function_path = f'{experiment_path}/original_function.png'
        aproximated_function_path = f'{experiment_path}/aproximated_function.png'
        loss_path = f'{experiment_path}/loss.png'
        
        print(f'Training time:\n\t{model.time:.2f} seconds\n\t{model.time / 60:.2f} minutes')

        mvn.plot(plot_n_samples, plot_samples, savefig=True, filepath=original_function_path)
        model.plot(plot_n_samples, plot_samples, savefig=True, filepath=aproximated_function_path)
        model.plot_loss(savefig=True, filepath=loss_path)
        
        loss_results[n_experiment, n_repetition] = model.dictionary_layer.losses[-1]
        time_results[n_experiment, n_repetition] = model.time
        
        plot_mean_std(loss_results, n_experiments, results_path, 'loss')
        plot_mean_std(time_results, n_experiments, results_path, 'time')
        plot_features_times(time_results, n_features, n_experiments, results_path)
        
        del model
        del X 
        del y
